In [ ]:
pip install transformers pandas torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

path = '/content/drive/My Drive/MACSS60000/FINAL_DATA/control_group_csv/you_should_know.csv'
df = pd.read_csv(path)

In [ ]:
df = df[:10000]

In [ ]:
sentences = ['You are a fucking shit', 'Wow, im so happy', 'Today is Monday']
df = pd.DataFrame({'selftext': sentences})

In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import torch
from torch.nn.functional import softmax

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=False,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten()
        }

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Move model to GPU
model = model.to('cuda')

# Prepare DataLoader
dataset = TextDataset(df['selftext'].tolist(), tokenizer)
loader = DataLoader(dataset, batch_size=16)

# Prediction function
# Prediction function with dynamic progress output
def predict_sentiment(data_loader, model):
    model.eval()
    probabilities = []
    total_batches = len(data_loader)

    with torch.no_grad():
        for batch_num, d in enumerate(data_loader, start=1):
            input_ids = d["input_ids"].to('cuda')
            attention_mask = d["attention_mask"].to('cuda')
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            batch_probs = softmax(outputs.logits, dim=1)
            probabilities.extend(batch_probs.cpu().numpy())
            print(f'Processing batch {batch_num}/{total_batches}...')
    return probabilities

df['sentiment'] = predict_sentiment(loader, model)
print(df)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Processing batch 1/1...
                 selftext                 sentiment
0  You are a fucking shit  [0.40748808, 0.59251195]
1        Wow, im so happy  [0.51503724, 0.48496273]
2         Today is Monday    [0.4513446, 0.5486554]


In [ ]:
sentences = ['You are a fucking shit', 'Wow, im so happy', 'Today is Monday', "The sentiment of this sentence is neutral", "This is my mobile phone number 12345678"]
df = pd.DataFrame({'selftext': sentences})

In [ ]:
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import torch
from torch.nn.functional import softmax

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.tokenizer = tokenizer
        self.texts = texts
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=False,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten()
        }

# Update to use the nlptown/bert-base-multilingual-uncased-sentiment model
tokenizer = BertTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')
model = BertForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

# Move model to GPU
model = model.to('cuda')

# Prepare DataLoader
dataset = TextDataset(df['selftext'].tolist(), tokenizer)
loader = DataLoader(dataset, batch_size=16)

# Prediction function with dynamic progress output
def predict_sentiment(data_loader, model):
    model.eval()
    probabilities = []
    total_batches = len(data_loader)

    with torch.no_grad():
        for batch_num, d in enumerate(data_loader, start=1):
            input_ids = d["input_ids"].to('cuda')
            attention_mask = d["attention_mask"].to('cuda')
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            # Apply softmax to convert logits to probabilities
            batch_probs = softmax(outputs.logits, dim=1)
            probabilities.extend(batch_probs.cpu().numpy())

            # Print progress
            print(f'Processing batch {batch_num}/{total_batches}...')

    return probabilities

# Perform sentiment analysis
df['sentiment_probabilities'] = predict_sentiment(loader, model)

df['predicted_rating'] = df['sentiment_probabilities'].apply(lambda x: x.argmax() + 1)  # Adding 1 because ratings start from 1 to 5

print(df[['selftext', 'predicted_rating']])


Processing batch 1/1...
                                    selftext  predicted_rating
0                     You are a fucking shit                 1
1                           Wow, im so happy                 5
2                            Today is Monday                 5
3  The sentiment of this sentence is natural                 5
4    This is my mobile phone number 12345678                 5


In [ ]:
!pip install openai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.4/227.4 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.4 MB/s eta 0:00:00


In [ ]:
! pip install openai==1.10.0
import os
import openai

openai.api_key = os.environ[""]
import getpass
OPENAI__KEY = getpass.getpass()

openai.api_key = OPENAI_API_KEY


from openai import OpenAI
client = OpenAI(
    api_key=openai.api_key,
)

In [ ]:
! pip install openai==1.10.0
import os
import openai
OPENAI_API_KEY = ""
openai.api_key = OPENAI_API_KEY
from openai import OpenAI
client = OpenAI(
    api_key=openai.api_key,
)
sentences = ['You are a fucking shit', 'Wow, im so happy', 'Today is Monday', "The sentiment of this sentence is natural", "This is my mobile phone number 12345678"]
df = pd.DataFrame({'selftext': sentences})
sentiment_scores = []
for index, row in df.iterrows():
    try:
        content = row['selftext']
        # Set up a message to request analysis from the API
        messages = [
            {"role": "system", "content": "You are now a professional psychologist specialized in detecting sentiment analysis score."},
            {"role": "user", "content": f"Analyze the following review for signs of sentiment analysis score, only give me the score between -1 to 1,do not give me other comments: '{content}'."}
        ]
        # Invoke the OpenAI API
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            temperature=0.7,
            max_tokens=100
        )
        # Add the analysis results to the list
        analysis = response.choices[0].message.content  # This is the correct way to access the content
        sentiment_scores.append(analysis)
    except Exception as e:
        # If an error occurs, add the error message to the list
        print(f"An error occurred for index {index}: {e}")
        sentiment_scores.append(f"Error: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.13.3
    Uninstalling openai-1.13.3:
      Successfully uninstalled openai-1.13.3


In [ ]:
sentiment_scores

['-0.8', '0.75', '0', '0.1', '0']

In [ ]:
sentences

['You are a fucking shit',
 'Wow, im so happy',
 'Today is Monday',
 'The sentiment of this sentence is natural',
 'This is my mobile phone number 12345678']